# 01 · Evaluación Traditional RAG
Evalúa el Traditional RAG (ChromaDB + MiniLM + Gemini) sobre UltraDomain con métricas RAGAS.

In [2]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../..'))
os.listdir()

['01_eval_traditional.ipynb', '00_DatasetLoading.ipynb']

## 1. Configuración del experimento

In [3]:
DOMINIO     = "cs"   # dominio de UltraDomain
N_LIBROS    = 1      # cuántos libros indexar
MAX_Q       = None      # preguntas a evaluar (None = todas)
SHUFFLE     = False  # True = libros aleatorios
CARPETA_DB  = "../chroma_db_eval_traditional"
RESULTS_DIR = "../../results/"

## 2. Cargar datos

In [4]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Guide to Java — James T. Streib (8 preguntas)
   Total Q&A: 8


In [4]:
# En vez de cargar el primer libro, cargar el que corresponde a las Q&A
# DOMINIO  = "cs"
# N_LIBROS = 1
# SHUFFLE  = False

# libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

# Verificar que coinciden
print(f"Libro indexado: {libros[0]['titulo']}")
print(f"Q&A son sobre: {qas[0]['titulo']}")

Libro indexado: Linux Kernel Networking
Q&A son sobre: Linux Kernel Networking


## 3. Inicializar y indexar el RAG

In [5]:
import shutil
from src.data_loader import load_and_split_text
from src.baselines.traditional_rag import TraditionalRAG

# Empezar limpio
shutil.rmtree(CARPETA_DB, ignore_errors=True)

rag = TraditionalRAG(persist_directory=CARPETA_DB)

for libro in libros:
    # Guardar texto en fichero temporal para load_and_split_text
    tmp = f"/tmp/{libro['context_id']}.txt"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(libro["texto"])
    splits = load_and_split_text(tmp)
    print(f"📖 {libro['titulo']} → {len(splits)} fragmentos")
    rag.index_documents(splits)

print("✅ Indexación completada")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📖 Linux Kernel Networking → 1849 fragmentos
✅ Indexación completada


## 4. Ejecutar evaluación

In [6]:
from src.evaluation.experiment import run_experiment 

result = await run_experiment(
    rag_type="traditional",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📚 Dominio: cs
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   Total Q&A: 8

🔍 Evaluando TRADITIONAL | cs | 8 preguntas
────────────────────────────────────────────────────────────
  [1/8] What are some advanced topics covered in the book related to Linux Ker...
  [2/8] How does the book address the implementation of IPv6 in comparison to ...
  [3/8] What are IP options and why might they be used?...
  [4/8] What role do netlink sockets play in Linux Kernel Networking?...
  [5/8] What is the significance of the ICMP protocol in Linux Kernel Networki...
  [6/8] Describe the structure and function of the IPv4 header....
  [7/8] What is the primary purpose of the Linux Kernel Networking stack as de...
  [8/8] Explain the process of IPv4 fragmentation and defragmentation....

📊 Calculando métricas RAGAS...
  RAGAS [1/8] What are some advanced topics covered in the book ...
    ⚠️ answer_correctness: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', '

## 5. Inspeccionar respuestas individuales

In [11]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")

❓ What is the difference between recording a macro and writing code from scratch in VBA?
✅ GT:  Recording a macro involves using the Macro Recorder to translate user actions into VBA code, while writing code from scratch involves manually typing ...
🤖 RAG: Puedes crear código VBA de dos maneras principales:

1.  **Grabando una macro:** Esto se hace usando la Grabadora de Macros, que está disponible en Wo...
⏱️  3.61s

❓ How do you assign a macro to a button on the Quick Access Toolbar in Word?
✅ GT:  To assign a macro to a button on the Quick Access Toolbar in Word, right-click the Quick Access Toolbar, choose Customize Quick Access Toolbar, select...
🤖 RAG: Para asignar una macro a un botón en la Barra de Acceso Rápido en Word, sigue estos pasos:

1.  Haz clic derecho en cualquier lugar de la Barra de Acc...
⏱️  5.07s

